### Chuyển video thành wav 1 kênh 16000 Hz

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from scripts.video_converter import convert_video_to_ready_wav_pydub


video_path = "../data/videos/Bai3_CongThucLuongGiac/RECORD_CongThucLuongGiac.mp4"
video_name = video_path.split("/")[-1].split(".")[0]

output_wav_path = f"../data/audio/test/{video_name}.wav"

convert_video_to_ready_wav_pydub(video_path, output_wav_path)

Đang đọc file video, vui lòng đợi...
✅ Đã chuyển đổi thành công: ../data/audio/test/RECORD_CongThucLuongGiac.wav


'../data/audio/test/RECORD_CongThucLuongGiac.wav'

### Chuyển audio đã convert thành văn bản transcribe

In [2]:
import math
import time
import os
from pydub import AudioSegment
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)


def transcribe_long_audio(input_wav_path: str, chunk_minutes: int = 10, model: str = "qwen/qwen3-asr-1.7b") -> list:
    """
    Cat audio thanh cac doan nho, gui len API de nhan dien va ghop ket qua.
    Ap dung co che Overlap (chong chong cheo) de chong mat chu o ranh gioi cat.
    Tra ve danh sach segments voi thoi gian: [{"start": float, "end": float, "text": str}, ...]
    """
    print(f"Dang tai file {input_wav_path}...")
    audio = AudioSegment.from_wav(input_wav_path)

    chunk_length_ms = chunk_minutes * 60 * 1000
    overlap_ms = 2 * 1000  # Chong cheo 2 giay
    total_length_ms = len(audio)
    total_chunks = math.ceil(total_length_ms / chunk_length_ms)

    print(f"Tong thoi luong: {total_length_ms / 1000 / 60:.2f} phut.")
    print(f"Se chia thanh {total_chunks} doan de xu ly.")

    all_segments = []

    for i in range(total_chunks):
        start_ms = i * chunk_length_ms
        end_ms = min(total_length_ms, (i + 1) * chunk_length_ms + overlap_ms)
        chunk_offset_sec = start_ms / 1000

        print(f"  Doan {i + 1}/{total_chunks} ({start_ms/1000:.0f}s - {end_ms/1000:.0f}s)... ", end="", flush=True)

        chunk = audio[start_ms:end_ms]
        temp_chunk_path = f"temp_chunk_{i}.wav"
        chunk.export(temp_chunk_path, format="wav", parameters=["-acodec", "pcm_s16le"])

        try:
            with open(temp_chunk_path, "rb") as audio_file:
                transcription = client.audio.transcriptions.create(
                    model=model,
                    file=audio_file,
                    language="vi",
                    response_format="verbose_json",
                    timestamp_granularities=["segment"]
                )
            
            if hasattr(transcription, "segments") and transcription.segments:
                for seg in transcription.segments:
                    all_segments.append({
                        "start": seg.start + chunk_offset_sec,
                        "end": seg.end + chunk_offset_sec,
                        "text": seg.text.strip()
                    })
            else:
                # Fallback: neu API khong tra ve segments, dung text thuan
                all_segments.append({
                    "start": chunk_offset_sec,
                    "end": end_ms / 1000,
                    "text": transcription.text.strip()
                })
            print("OK")
        except Exception as e:
            print(f"LOI: {e}")
        finally:
            if os.path.exists(temp_chunk_path):
                os.remove(temp_chunk_path)

        # Nghii 1 giay giua cac doan de tranh rate limit
        if i < total_chunks - 1:
            time.sleep(1)

    total_chars = sum(len(s["text"]) for s in all_segments)
    print(f"\nHoan thanh! Tong so ky tu: {total_chars}, so segment: {len(all_segments)}")
    return all_segments


def format_time(seconds: float) -> str:
    """Convert seconds to HH:MM:SS format"""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"


segments = transcribe_long_audio(output_wav_path, chunk_minutes=10)

# Hien thi ket qua voi thoi gian
print("\n=== TRANSCRIPT VOI THOI GIAN ===\n")
for seg in segments:
    print(f"[{format_time(seg['start'])} - {format_time(seg['end'])}] {seg['text']}")

Dang tai file ../data/audio/test/RECORD_CongThucLuongGiac.wav...
Tong thoi luong: 167.08 phut.
Se chia thanh 17 doan de xu ly.
  Doan 1/17 (0s - 602s)... OK
  Doan 2/17 (600s - 1202s)... OK
  Doan 3/17 (1200s - 1802s)... OK
  Doan 4/17 (1800s - 2402s)... OK
  Doan 5/17 (2400s - 3002s)... OK
  Doan 6/17 (3000s - 3602s)... OK
  Doan 7/17 (3600s - 4202s)... OK
  Doan 8/17 (4200s - 4802s)... OK
  Doan 9/17 (4800s - 5402s)... OK
  Doan 10/17 (5400s - 6002s)... OK
  Doan 11/17 (6000s - 6602s)... OK
  Doan 12/17 (6600s - 7202s)... OK
  Doan 13/17 (7200s - 7802s)... OK
  Doan 14/17 (7800s - 8402s)... OK
  Doan 15/17 (8400s - 9002s)... OK
  Doan 16/17 (9000s - 9602s)... OK
  Doan 17/17 (9600s - 10025s)... OK

Hoan thanh! Tong so ky tu: 106774, so segment: 1529

=== TRANSCRIPT VOI THOI GIAN ===

[00:00 - 00:00] thể record.
[00:03 - 00:06] nào hiện tại là chúng ta đang qua cái phần buổi thứ ba.
[00:07 - 00:28] của lớp nền tảng toán cho vật lý hiện tại là thầy đã dạy xong cho bạn về nội dung về ph

In [3]:
# segments da duoc gan o cell tren
# Chuyen doi segments thanh text de su dung cho LLM
audio_bai_giang = " ".join([f"<{format_time(s['start'])} - {format_time(s['end'])}>{s['text']}" for s in segments])

In [4]:
print(audio_bai_giang)

<00:00 - 00:00>thể record. <00:03 - 00:06>nào hiện tại là chúng ta đang qua cái phần buổi thứ ba. <00:07 - 00:28>của lớp nền tảng toán cho vật lý hiện tại là thầy đã dạy xong cho bạn về nội dung về phần vector phần nhân vô hướng và phần nhân có hướng rồi có đúng không và thêm nữa là thầy cũng đã gửi cho bạn các cái phần file bài tập về nhà để cho chúng ta làm thì những cái phần file bài tập về nhà thì thầy nhắc lại một lần nữa đó là những cái phần file đó nội dung nó không hề khó <00:29 - 00:39>ở đây là mục tiêu của chúng ta, chúng ta cần phải làm những phần bài tập về nhà đấy để các bạn gọi là học thuộc công thức trước đã, đúng không? Thế nên là thầy chọn những cái bài tập mà nó chỉ ở mức độ cơ bản thôi. <00:39 - 00:49>thì chúng ta học thúng thức thông qua việc chúng ta làm nhiều bài tập trách nhiệm và những phần file đó thì các bạn nếu mà các bạn dành thời gian ra làm thì thầy nghĩ rằng chỉ cần khoảng tầm một đến hai tiếng là các bạn xong một file rồi <00:51 - 00:54>thấy nghĩ rằng là

### Dùng LLM để tách thành các chủ đề con

In [5]:
import instructor
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from scripts.schemas import TopicChunkLst

load_dotenv()  # Load environment variables from .env file

api_key = os.getenv("OPENROUTER_API_KEY")

SYSTEM_PROMPT = """
Bạn là chuyên gia phân tích nội dung giáo dục.
Bạn sẽ nhận được transcript của một bài giảng, mỗi dòng có định dạng [MM:SS] nội dung.

Hãy tách ra thành các CHỦ ĐỀ LỚN được nói tới trong bài giảng.
Mỗi chủ đề nên có thời lượng ít nhất 20 phút. Các nội dung nhỏ lẻ liên quan đến nhau nên được gộp thành một chủ đề lớn.

Ví dụ:
- "Công thức lượng giác cơ bản, công thức cộng, nhân đôi, hạ bậc" nên là MỘT chủ đề
- "Định lý sin" là một chủ đề riêng
- "Định lý côsin" là một chủ đề riêng
- "Bài tập ứng dụng" có thể gộp nhiều bài tập liên quan vào một chủ đề

Với mỗi chủ đề, hãy xác định thời gian bắt đầu và kết thúc dựa trên timestamp trong transcript.
Tóm tắt nội dung mỗi chủ đề một cách súc tích.
"""

# Wrap OpenAI client with instructor
client = instructor.from_openai(
    OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key
    )
)

# Use instructor to extract structured TopicChunkLst
topic_chunks = client.chat.completions.create(
    model="deepseek/deepseek-v4.1-flash",
    response_model=TopicChunkLst,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Transcribe của bài giảng:\n\n{audio_bai_giang}"}
    ],
    temperature=0.2,
)


In [6]:
# Hien thi ket qua
for i, chunk in enumerate(topic_chunks.topics, 1):
    print(f"\n--- Chu de {i}: {chunk.topic} ---")
    print(f"Thoi gian: {chunk.start_time} - {chunk.end_time}")
    print(f"Tom tat: {chunk.summary}")


--- Chu de 1: Giới thiệu: Vì sao cần học hệ thức lượng và công thức lượng giác ---
Thoi gian: 00:00 - 14:22
Tom tat: Thầy nhắc lại nội dung buổi trước (vector, nhân vô hướng, nhân có hướng) và bài tập về nhà (mức cơ bản, mục tiêu học thuộc công thức). Giải thích lý do học: (1) ứng dụng trong toán học để tính cạnh, góc trong tam giác bất kỳ; (2) ứng dụng trong vật lý như phân tích vectơ lực (ví dụ vật trượt trên mặt phẳng nghiêng), giải phương trình dao động điều hòa (con lắc đơn, con lắc lò xo), dòng điện xoay chiều, giao thoa sóng. Nhấn mạnh đây là phần có nhiều công thức nhất nhưng cũng dễ học nhất nếu hiểu bản chất và chứng minh thay vì học vẹt.

--- Chu de 2: Hệ thức lượng trong tam giác: Định lý côsin và định lý sin ---
Thoi gian: 14:22 - 34:50
Tom tat: Ôn tập các hệ thức lượng trong tam giác vuông đã học ở cấp hai (b²=b'·a, c²=c'·a, a²=b²+c², ah=bc, 1/h²=1/b²+1/c², h²=b'·c'). Mở rộng sang tam giác bất kỳ: trình bày định lý côsin (a²=b²+c²−2bc·cosα và các công thức tương tự cho b

In [7]:
# Xem du lieu thô
print(topic_chunks.model_dump_json(indent=2))

{
  "topics": [
    {
      "start_time": "00:00",
      "end_time": "14:22",
      "topic": "Giới thiệu: Vì sao cần học hệ thức lượng và công thức lượng giác",
      "summary": "Thầy nhắc lại nội dung buổi trước (vector, nhân vô hướng, nhân có hướng) và bài tập về nhà (mức cơ bản, mục tiêu học thuộc công thức). Giải thích lý do học: (1) ứng dụng trong toán học để tính cạnh, góc trong tam giác bất kỳ; (2) ứng dụng trong vật lý như phân tích vectơ lực (ví dụ vật trượt trên mặt phẳng nghiêng), giải phương trình dao động điều hòa (con lắc đơn, con lắc lò xo), dòng điện xoay chiều, giao thoa sóng. Nhấn mạnh đây là phần có nhiều công thức nhất nhưng cũng dễ học nhất nếu hiểu bản chất và chứng minh thay vì học vẹt."
    },
    {
      "start_time": "14:22",
      "end_time": "34:50",
      "topic": "Hệ thức lượng trong tam giác: Định lý côsin và định lý sin",
      "summary": "Ôn tập các hệ thức lượng trong tam giác vuông đã học ở cấp hai (b²=b'·a, c²=c'·a, a²=b²+c², ah=bc, 1/h²=1/b²+1/c², h

### Chia video theo từng chủ đề

In [8]:
from scripts.video_converter import split_video_by_topics

# Chia video theo các chủ đề đã xác định
output_videos = split_video_by_topics(
    video_path=video_path,
    topics=topic_chunks.topics,
    output_dir=None  # None = cùng folder với video gốc
)

# In danh sách video đã tạo
print("=== DANH SACH VIDEO DA TAO ===")
for v in output_videos:
    print(v)

Đang chia video thành 6 phần...
  [1/6] 00:00 - 14:22: Giới thiệu: Vì sao cần học hệ thức lượng và công t...
    ✅ Thành công: RECORD_CongThucLuongGiac_1.mp4
  [2/6] 14:22 - 34:50: Hệ thức lượng trong tam giác: Định lý côsin và địn...
    ✅ Thành công: RECORD_CongThucLuongGiac_2.mp4
  [3/6] 34:50 - 51:22: Đơn vị góc radian và các cung lượng giác liên quan...
    ✅ Thành công: RECORD_CongThucLuongGiac_3.mp4
  [4/6] 51:22 - 01:27:58: Đường tròn lượng giác và chứng minh các công thức ...
    ✅ Thành công: RECORD_CongThucLuongGiac_4.mp4
  [5/6] 01:27:58 - 01:58:25: Các công thức lượng giác cơ bản: cộng, nhân, hạ bậ...
    ✅ Thành công: RECORD_CongThucLuongGiac_5.mp4
  [6/6] 01:58:25 - 02:46:55: Bài tập ứng dụng hệ thức lượng và công thức lượng ...
    ✅ Thành công: RECORD_CongThucLuongGiac_6.mp4

Hoàn thành! Đã tạo 6/6 video con.
=== DANH SACH VIDEO DA TAO ===
../data/videos/Bai3_CongThucLuongGiac/splitted/RECORD_CongThucLuongGiac_1.mp4
../data/videos/Bai3_CongThucLuongGiac/splitted/RECORD